# Introduction

In part 1 of this assessment, you will complete several requested SQL queries in order to extract data, analyze, and provide insights from a single provided SQL database. You will also visualize the key results of 3 of these queries. There are also several 'Reflection' questions that ask you to write out a text based answer in the provided markdown cell. Following the guided question and answer section, in part 2 you will explore a second dataset on your own using SQL in order to conduct a preliminary analysis. You will be asked to produce a very short slide presentation highlighting the work you did for this second section.

## Objectives
You will be able to:
- Interpret "word problems" and translate them into SQL queries
- Decide and perform whichever type of JOIN is best for retrieving desired data
- Use GROUP BY statements to apply aggregate functions like COUNT, MAX, MIN, and SUM
- Use the HAVING clause to compare different aggregates
- Write subqueries to decompose complex queries
- Visualize data using matplotlib, seaborn, or pandas
- Choose the correct chart type based on the given data


## Part 1: Guided SQL Queries

### Your Task: Querying a Customer Database

![toy car picture](images/toycars.jpg)


### Business Understanding
Your employer sells wholesale miniature models of products such as classic cars, motorcycles, and planes. They want you to pull several reports on different segments of their past customers, in order to better understand past sales as well as determine which customers will receive promotional material. They are also interested in investigating which products have performed the best, as well as having several smaller asks.

In addition to providing the requested data from the SQL database you have also been asked to create some basic visuals to display some of the more insightful information. It is up to your discretion to choose the correct plot/chart type for the data in question. **Questions that want you to visualize the results will be explicitly marked**.

### Data Understanding
You may remember this database from a previous lab. As a refresher, here's the ERD diagram for this database:

![ERD picture](images/ERD.png)

The queries you are asked to write will become more complex over the course of the lab.



### Getting Started
For this assessment you are expected to make use of both sqlite3 and the Pandas libraries in order to write, execute, and return SQL queries as a Pandas DataFrame. Assign each returned answer as its own explicit variable.

For the visualization piece you are expected to utilize either Pandas, Seaborn, or Matplotlib to create your visuals. Make sure you are providing verbose labels and titles according to the data you are being asked to visualize. Do not worry too much about choosing a 'style' or 'context' instead focus on conveying the requested information correctly.

### Step 1: Connect to Data

In the cell below
- Import the necessary libraries
- Establish a connection to the database data.sqlite

In [ ]:
# Imports
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Create connection to database
conn = sqlite3.connect('data.sqlite')

### Step 2: Limited Edition California Product
The California sales rep team is interested in running promotional material for a new limited edition model they are releasing based on the famous San Francisco Cable Cars. This product will only be available to customer stores based in California and given its high price value they want to first target promotional material to existing California customers with a high credit limit. Upon communicating with the accounting department, a credit limit of over 25,000 is considered to be high. 

Execute a SQl query that returns which customers the sales rep team wants to market to first.
*Hint*: Make sure creditLimit is numeric.

In [ ]:
# creditLimit is stored as text, so CAST it to a number before comparing
ca_high_credit_df = pd.read_sql("""
SELECT customerNumber, customerName, contactFirstName, contactLastName,
       city, state, CAST(creditLimit AS REAL) AS creditLimit
FROM customers
WHERE state = 'CA'
  AND CAST(creditLimit AS REAL) > 25000
ORDER BY creditLimit DESC;
""", conn)
ca_high_credit_df

### Step 3: International Collectable Campaign

The international sales rep team has reached out to you to help them identify partners for a 'Collectable' marketing campaign that highlights the potential collectors value in purchasing these model kits. They want to try and promote a 'collect them all' mentality. The team had a great idea to partner with any of their international customers (non-US) who have "Collect" in their name as a tie in to the larger theme.

Execute a SQL that returns the customers in question.

In [ ]:
intl_collectable_df = pd.read_sql("""
SELECT customerNumber, customerName, city, country
FROM customers
WHERE country != 'USA'
  AND customerName LIKE '%Collect%'
ORDER BY country, customerName;
""", conn)
intl_collectable_df

## Reflection Question:

Describe the WHERE clause you used in the above query to a non-technical manager who wants to be ensured that you are properly filtering and only selecting the requested data. How is the operator and conditional expression you are using acting to accomplish this?

## Answer

The WHERE clause is a two-part checklist that every customer row has to pass before it is allowed into the results. Both parts must be true (that is what AND means):

1. `country != 'USA'` keeps only customers whose country is anything other than the United States, which gives us the international customers.
2. `customerName LIKE '%Collect%'` keeps only customers whose name contains the text "Collect". The `%` signs are wildcards meaning "anything can come before or after", so it matches "UK Collectables, Ltd.", "Australian Collectors, Co." and "Clover Collections, Co." alike. This is why one short pattern catches every spelling variation (Collectables, Collectors, Collections) that fits the "collect them all" theme.

A customer only shows up if it passes both checks, so a US customer such as "West Coast Collectables Co." is correctly left out even though it has "Collect" in its name.

### Step 4: USA Credit and Inventory Policy - Visual Required
The USA based product team is planning to adjust its credit policies and inventory allocation strategy based on the average credit limit of its customers. They would like to target this strategy at a state level with several goals in mind. 
1. Optimize inventory distribution:
    - States with higher average credit limits might be able to place larger orders, justifying priority in inventory allocation.
    - This could help ensure that states with more purchasing power always have products in stock.
2. Tailor credit policies:
    - Adjust credit limits for new customers based on the state average.
    - Identify states where they might be too conservative or too liberal with credit limits.
3. Target marketing and sales efforts:
    - Focus promotional campaigns on states with higher credit limits, potentially leading to larger orders.
    - Develop strategies to increase sales in states with lower average credit limits.

Execute a SQl query that returns the information required to address this ask.

In [ ]:
avg_credit_by_state_df = pd.read_sql("""
SELECT state,
       COUNT(*) AS num_customers,
       ROUND(AVG(CAST(creditLimit AS REAL)), 2) AS avg_credit_limit
FROM customers
WHERE country = 'USA'
GROUP BY state
ORDER BY avg_credit_limit DESC;
""", conn)
avg_credit_by_state_df

Once you have the information returned in a dataframe, select an appropriate visualization to represent this data. You are welcome to utilize matplotlib, seaborn, or pandas plotting to produce your visual. Ensure that it has a verbose title and axis labels!

In [ ]:
# Produce a visual to represent the average credit limit by state
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(avg_credit_by_state_df['state'], avg_credit_by_state_df['avg_credit_limit'],
               color=plt.cm.viridis(avg_credit_by_state_df['avg_credit_limit'] / avg_credit_by_state_df['avg_credit_limit'].max()))
overall = avg_credit_by_state_df['avg_credit_limit'].mean()
ax.axhline(overall, color='red', linestyle='--', label=f'Average of state averages (${overall:,.0f})')
ax.bar_label(bars, labels=[f"${v:,.0f}\n(n={n})" for v, n in zip(avg_credit_by_state_df['avg_credit_limit'], avg_credit_by_state_df['num_customers'])], fontsize=8)
ax.set_title('Average Customer Credit Limit by US State (n = number of customers)')
ax.set_xlabel('US State')
ax.set_ylabel('Average Credit Limit (USD)')
ax.legend()
plt.tight_layout()
plt.show()

### Step 5: Top Customers - Visual Required
The company is approaching its 10 year anniversary and wants to acknowledge and thank its top customers with personalized communication. They have asked you to determine the top 10 customers based on the total amount of payments made, making sure to return the customer name for clarity. 

Execute a SQl query that returns the information required to address this ask.


In [ ]:
top10_customers_df = pd.read_sql("""
SELECT c.customerName,
       ROUND(SUM(p.amount), 2) AS total_payments
FROM customers c
JOIN payments p ON c.customerNumber = p.customerNumber
GROUP BY c.customerNumber, c.customerName
ORDER BY total_payments DESC
LIMIT 10;
""", conn)
top10_customers_df

Once you have the information returned in a dataframe, select an appropriate visualization to represent this data. You are welcome to utilize matplotlib, seaborn, or pandas plotting to produce your visual. Ensure that it has a verbose title and axis labels!

In [ ]:
# Produce a visual to represent the top ten customers in terms of total payments
fig, ax = plt.subplots(figsize=(10, 6))
d = top10_customers_df.iloc[::-1]
bars = ax.barh(d['customerName'], d['total_payments'], color=plt.cm.Blues(0.35 + 0.6 * d['total_payments'] / d['total_payments'].max()))
ax.bar_label(bars, labels=[f"${v:,.0f}" for v in d['total_payments']], padding=3)
ax.set_xlim(0, d['total_payments'].max() * 1.15)
ax.set_title('Top 10 Customers by Total Payments Made')
ax.set_xlabel('Total Payments (USD)')
ax.set_ylabel('Customer Name')
plt.tight_layout()
plt.show()

### Step 6: Top Customer + Product Quantities
The product team is running an analysis on popular and common products sold to each customer in order to try and determine what new products they should be looking at to include in their catalog. This data will also be used by individual sales reps to recommend similar products to each customer next time they place an order. 

They have asked you to query information, for each customer, about any product they have purchased 10 or more units of. In addition they would like the full set of data to be sorted in ascending order by the total amount purchased.

Execute a SQl query that returns the information required to address this ask.

Hint: For this one, you'll need to make use of HAVING, GROUP BY, and ORDER BY — make sure you get the order of them correct!

In [ ]:
customer_product_qty_df = pd.read_sql("""
SELECT c.customerNumber,
       c.customerName,
       p.productCode,
       p.productName,
       SUM(od.quantityOrdered) AS total_quantity
FROM customers c
JOIN orders o ON c.customerNumber = o.customerNumber
JOIN orderdetails od ON o.orderNumber = od.orderNumber
JOIN products p ON od.productCode = p.productCode
GROUP BY c.customerNumber, c.customerName, p.productCode, p.productName
HAVING SUM(od.quantityOrdered) >= 10
ORDER BY total_quantity ASC;
""", conn)
customer_product_qty_df

### Step 7: Product Analysis - Visual Required

The product team is looking into the demand across its different product lines. They are conducting a comprehensive review of its product portfolio and inventory management strategies. You have been asked to query data pertaining to each different product line, that contains the total quantity ordered and the total number of products for each respective product line. By examining the number of products and total quantity ordered for each product line, the company aims to:
1. Optimize product mix:
    - Identify which product lines have the most diverse offerings (high number of products)
    - Determine which lines are most popular (high total quantity ordered)
    - Compare if lines with more products necessarily lead to more orders
2. Improve inventory management:
    - Adjust stock levels based on the popularity of each product line
    - Identify potential overstocking in lines with low order quantities
    - Ensure adequate variety in high-performing product lines
3. Adjust marketing strategy:
    - Focus promotional efforts on product lines with high potential (many products but lower order quantities)
    - Capitalize on the popularity of high-performing lines in marketing campaigns
4. Advise Product development:
    - Invest in expanding product ranges for lines with high order quantities
    - Consider phasing out or revamping product lines with low numbers of products and low order quantities

Hint: Think about how you can and might have to utilize SQL DISTINCT statement

Execute a SQl query that returns the information required to address this ask.

In [ ]:
product_line_df = pd.read_sql("""
SELECT pl.productLine,
       COUNT(DISTINCT p.productCode) AS num_products,
       SUM(od.quantityOrdered) AS total_quantity_ordered
FROM productlines pl
JOIN products p ON pl.productLine = p.productLine
JOIN orderdetails od ON p.productCode = od.productCode
GROUP BY pl.productLine
ORDER BY total_quantity_ordered DESC;
""", conn)
product_line_df

Once you have the information returned in a dataframe, select an appropriate visualization to represent the relationship between total quantity ordered and the number of products in order to perform a preliminary investigation into the question of if more products lead to more orders. You are welcome to utilize matplotlib, seaborn, or pandas plotting to produce your visual. Ensure that it has a verbose title and axis labels!

In [ ]:
# Produce a visual to represent the the relation between number of products and the total amount ordered
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=product_line_df, x='num_products', y='total_quantity_ordered', s=120, ax=ax)
for _, r in product_line_df.iterrows():
    ax.annotate(r['productLine'], (r['num_products'], r['total_quantity_ordered']),
                textcoords='offset points', xytext=(8, 4))
ax.set_title('Number of Products vs. Total Quantity Ordered by Product Line')
ax.set_xlabel('Number of Distinct Products in Product Line')
ax.set_ylabel('Total Quantity Ordered (units)')
plt.tight_layout()
plt.show()

## Reflection Question:

Please explain your choice in the type of visual you used in order to highlight and represent the data from the above query. In a non-technical manner explain why that chart type makes sense for the information being conveyed. What does this visual convey in the context of the question it was asked for?

## Answer

I used a scatter plot because the question is whether two numbers move together: how many products a line has, and how many units customers ordered from it. A scatter plot gives each product line one dot, placed by those two numbers, so you can see at a glance whether dots climb from bottom-left to top-right. A bar chart would only show one number at a time and hide the relationship.

The dots do climb steadily: Classic Cars has the most products (37) and the most units ordered (about 35,600), while Trains has the fewest products (3) and the fewest units (about 2,800). So, in this data, lines with more products do tend to sell more units overall. One caution: there are only seven product lines, and a line with more products naturally has more items to sell, so this is a preliminary signal and not proof that adding products causes more orders.

### Step 8: Remote Offices
Upper management is considering a shift to hybrid and remote work for certain locations and roles. They have tasked you with providing them data about employees who work in any office that has fewer than 5 total employees so they can better understand how to support those employees remotely when offices are shut down. 

Be sure to include information about the employees job and supervisor so management can adjust everyone to remote work properly.

Hint: Utilize a subquery to find the relevant offices

Execute a SQl query that returns the information required to address this ask.

In [ ]:
remote_office_employees_df = pd.read_sql("""
SELECT e.employeeNumber,
       e.firstName,
       e.lastName,
       e.jobTitle,
       e.email,
       o.officeCode,
       o.city AS office_city,
       o.country AS office_country,
       s.firstName || ' ' || s.lastName AS supervisor_name,
       s.jobTitle AS supervisor_title
FROM employees e
JOIN offices o ON e.officeCode = o.officeCode
LEFT JOIN employees s ON e.reportsTo = s.employeeNumber
WHERE e.officeCode IN (
    SELECT officeCode
    FROM employees
    GROUP BY officeCode
    HAVING COUNT(*) < 5
)
ORDER BY o.officeCode, e.lastName;
""", conn)
remote_office_employees_df

## Reflection Question:

Describe how you decided on the subquery that you used in the query above? This answer can be technically in nature, describing your thought process in how the main query is utilizing the subquery to return the correct data.

## Answer

The ask has two steps: first work out which offices are small, then list the employees in those offices. The first step needs an aggregate (headcount per office) and the second needs row-level employee details, and an aggregate cannot be mixed into the same WHERE as row-level columns. So I put step one in a subquery: `SELECT officeCode FROM employees GROUP BY officeCode HAVING COUNT(*) < 5`. It groups employees by office and uses HAVING (not WHERE, because the condition is on a count) to keep offices with fewer than 5 people. The outer query then uses `WHERE e.officeCode IN (subquery)` to keep only employees in those offices.

Since the subquery runs on the full employees table, it counts everyone in each office before the filtering happens, so the headcount is correct. To give management supervisor information, I self-joined employees (`e.reportsTo = s.employeeNumber`) with a LEFT JOIN, so an employee without a supervisor would not be dropped, and joined offices for the city and country.

### Step 9: Close the Connection

Now that you are finished executing your queries and retrieving the required information you always want to make sure to close the connection to your database.

In [ ]:
conn.close()

### End of Guided Section
In this initial portion of the assessment, you produced several data queries and visualizations for a model company, mainly focused around its customer and product data. You wrote and engineered specific SQL queries to address pertinent questions and asks from the company. Along the way, you utilized many of the major concepts and keywords associated with SQL SELECT queries: FROM, WHERE, GROUP BY, HAVING, ORDER BY, JOIN, SUM, COUNT, and AVG.

## Part 2: Exploratory Analysis with SQL
In this open-ended exploratory section, you will analyze real-world data from the movie industry. As a data analyst, you have the freedom to investigate questions and topics that intrigue you within this dataset. The database schema and Entity-Relationship Diagram (ERD) are provided below for your reference. A general overview and instructions are also provided below.

In [3]:
# Run this cell without changes
import zipfile

zip_file_path = 'im.db.zip'
extract_to_path = './'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)

# Connection
conn4 = sqlite3.connect('im.db')

# Schema
schema_df = pd.read_sql("""
SElECT * FROM sqlite_master                        
""", conn4)
schema_df

,type,name,tbl_name,rootpage,sql
0,table,movie_basics,movie_basics,2,"CREATE TABLE ""movie_basics"" (\n""movie_id"" TEXT..."
1,table,directors,directors,3,"CREATE TABLE ""directors"" (\n""movie_id"" TEXT,\n..."
2,table,known_for,known_for,4,"CREATE TABLE ""known_for"" (\n""person_id"" TEXT,\..."
3,table,movie_akas,movie_akas,5,"CREATE TABLE ""movie_akas"" (\n""movie_id"" TEXT,\..."
4,table,movie_ratings,movie_ratings,6,"CREATE TABLE ""movie_ratings"" (\n""movie_id"" TEX..."
5,table,persons,persons,7,"CREATE TABLE ""persons"" (\n""person_id"" TEXT,\n ..."
6,table,principals,principals,8,"CREATE TABLE ""principals"" (\n""movie_id"" TEXT,\..."
7,table,writers,writers,9,"CREATE TABLE ""writers"" (\n""movie_id"" TEXT,\n ..."


## The Data

![movie ERD](images/movie_data_erd.jpeg)
### Database Content:

- Source: IMDB
- Time Range: Movies released between 2010 and 2019
- Note: Exclude any movies with a start_year after 2019 as this data is not current or accurate

Available Data Categories:
- Genre
- Runtime
- Personnel (writers, directors, actors)
- Movie ratings

### Objectives:

Initial Exploration:
- Use SQL in combination with Pandas to explore the database
- Identify interesting trends, patterns, or relationships in the data

Business Question Formulation:
- Develop at least one substantial business question for deeper analysis
- Ensure the question is relevant, specific, and can be addressed with the available data

Data Cleaning Assessment:
- Identify potential data cleaning tasks necessary for your deeper analysis
- Note: You are not required to perform the cleaning, only to recognize and list the necessary tasks

Null Value Handling:
- Be aware that the dataset contains null values in certain fields
- Exclude these null values from your exploration
- Do not attempt to input or fill in missing information

### Deliverables:

You need to produce a short slide presentation (3-5 slides) that highlights the three key deliverables below. Utilize a data visualization to support the second deliverable.

1. A summary of your initial data exploration findings
    - Can be bulleted or sentence form
2. At least one well-formulated business question for further analysis
    - Should stem from a relevant trend or pattern your initial exploration identified
3. A list of potential data cleaning tasks identified during your exploration
    - This can and should include things like data normalization/standardization and null handling

Tips for Success:

Begin with broad exploratory queries to understand the data's scope and content. Then focus on honing in on interesting relationships between different data categories. Consider industry trends, audience preferences, or financial aspects when formulating your business question. Pay attention to data quality issues, inconsistencies, or limitations that might affect your analysis. Remember, the goal is to demonstrate your analytical thinking and ability to derive meaningful insights from complex datasets. Good luck with your exploration!

NOTE: You do not need to explore every aspect of this database. Find something that you think is interesting or relevant about the data and focus your exploration there.

In [ ]:
# Begin your code here
# NOTE: persons/principals etc. are large, so keep queries targeted.
conn4 = sqlite3.connect('im.db')   # (safe to re-run; same database as above)

# 1. Scope of the data: how many movies, which years, how many nulls?
scope_df = pd.read_sql("""
SELECT COUNT(*) AS total_movies,
       MIN(start_year) AS min_year,
       MAX(start_year) AS max_year,
       SUM(start_year > 2019) AS after_2019,
       SUM(runtime_minutes IS NULL) AS null_runtime,
       SUM(genres IS NULL) AS null_genres
FROM movie_basics;
""", conn4)
display(scope_df)

# 2. Pull movies 2010-2019 that have a rating, a genre, and a runtime (nulls excluded, not filled in)
movies_df = pd.read_sql("""
SELECT b.movie_id, b.primary_title, b.start_year, b.runtime_minutes, b.genres,
       r.averagerating, r.numvotes
FROM movie_basics b
JOIN movie_ratings r ON b.movie_id = r.movie_id
WHERE b.start_year BETWEEN 2010 AND 2019
  AND b.genres IS NOT NULL
  AND b.runtime_minutes IS NOT NULL;
""", conn4)
print(f"{len(movies_df):,} rated movies with genre and runtime (2010-2019)")

# 3. Rating and engagement by genre. genres is a comma-separated string, so split it into one row per genre.
genre_df = movies_df.assign(genre=movies_df['genres'].str.split(',')).explode('genre')
genre_summary = (genre_df.groupby('genre')
                 .agg(num_movies=('movie_id', 'count'),
                      avg_rating=('averagerating', 'mean'),
                      median_votes=('numvotes', 'median'))
                 .query('num_movies >= 500')          # ignore tiny genres
                 .sort_values('avg_rating', ascending=False)
                 .round(2))
display(genre_summary)

# 4. Does rating or engagement change over time, and with runtime?
year_summary = (movies_df.groupby('start_year')
                .agg(num_movies=('movie_id', 'count'), avg_rating=('averagerating', 'mean'))
                .round(2))
display(year_summary)
display(movies_df[['averagerating', 'numvotes', 'runtime_minutes']].corr(method='spearman').round(2))

# 5. Visual: average rating by genre, shaded by audience engagement
plot_df = genre_summary.reset_index().sort_values('avg_rating')
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(plot_df['genre'], plot_df['avg_rating'],
               color=plt.cm.Blues(plot_df['median_votes'] / plot_df['median_votes'].max() * 0.8 + 0.2))
for b, v in zip(bars, plot_df['median_votes']):
    ax.text(b.get_width() + 0.05, b.get_y() + b.get_height()/2, f"{v:,.0f} median votes", va='center', fontsize=8)
ax.set_title('Average IMDB Rating by Genre, 2010-2019 (darker = more audience votes)')
ax.set_xlabel('Average Rating (1-10)')
ax.set_ylabel('Genre')
ax.set_xlim(0, 9.5)
plt.tight_layout()
plt.show()

In [ ]:
# Evidence for the data cleaning list (counts only; nothing is cleaned or filled in)
cleaning_checks_df = pd.read_sql("""
SELECT
  (SELECT COUNT(*) FROM movie_basics WHERE start_year > 2019)                              AS start_year_after_2019,
  (SELECT MAX(start_year) FROM movie_basics)                                                AS max_start_year,
  (SELECT COUNT(*) FROM movie_basics WHERE runtime_minutes IS NULL)                         AS null_runtime,
  (SELECT COUNT(*) FROM movie_basics WHERE genres IS NULL)                                  AS null_genres,
  (SELECT COUNT(*) FROM movie_basics) - (SELECT COUNT(*) FROM movie_ratings)                 AS movies_without_rating,
  (SELECT MIN(runtime_minutes) FROM movie_basics)                                           AS min_runtime,
  (SELECT MAX(runtime_minutes) FROM movie_basics)                                           AS max_runtime,
  (SELECT COUNT(*) FROM movie_ratings WHERE numvotes < 10)                                  AS ratings_under_10_votes,
  (SELECT COUNT(*) FROM (SELECT 1 FROM movie_basics GROUP BY primary_title, start_year HAVING COUNT(*) > 1)) AS duplicate_title_year_pairs,
  (SELECT COUNT(*) FROM movie_akas WHERE region IS NULL)                                    AS akas_null_region;
""", conn4)
cleaning_checks_df.T.rename(columns={0: 'count'})


## Part 2 Findings (summarized in the slide deck)

**Exploration findings**
- 146,144 titles in `movie_basics`; 73,856 have a rating. 1,063 titles have a start_year after 2019 (up to 2115) and are excluded.
- Documentary is the biggest genre (about 16,400 rated movies with a genre and runtime) and has the highest average rating (7.32), but a median of only 21 votes per film.
- Action (5.78), Thriller (5.59), Sci-Fi (5.43) and Horror (4.93) have the lowest ratings, yet Action (242) and Crime (about 204) draw the most votes per film.
- Average rating drifts up slightly across the decade (6.27 in 2010 to 6.38 in 2018); 2019 looks higher (6.65) but is a partial year.
- Rating has almost no relationship with runtime (Spearman -0.07); rating and votes are weakly negatively related (-0.25).

**Business question**
Which genres give a new studio the best balance of audience approval (average rating) and audience reach (number of votes) for 2010-2019 releases, and does that balance hold when runtime is taken into account?

**Data cleaning tasks**
1. Exclude start_year > 2019 (1,063 rows, including a 2115 entry).
2. Handle nulls: runtime_minutes (31,739), genres (5,408), unrated movies (72,288 have no rating row); exclude, do not fill.
3. Split the comma-separated `genres` column into one row per genre (or a separate table); normalize genre names.
4. Remove runtime outliers (values as low as 1 minute and as high as 51,420) and decide on a minimum runtime for "feature film".
5. Set a minimum numvotes threshold for ratings to be reliable (12,146 rated titles have fewer than 10 votes).
6. Resolve duplicate titles (same primary_title and start_year appears in 1,870 cases) and standardize primary_title vs original_title.
7. Standardize data types (birth_year/death_year and some numeric fields are stored as text) and clean `movie_akas` (53,293 null regions, inconsistent region codes like XWW).